# Ingest: NYC Yellow Taxi 2024

## Overview

This notebook loads all 12 months of NYC Yellow Taxi trip data (January through December 2024) from parquet files into PostgreSQL. The raw data is ingested into the trips_raw table without any cleaning or transformation. This preserves the original data for reproducibility and allows cleaning to be done in the next phase.

We also load the TLC taxi zone lookup table, which maps LocationID codes to borough and zone names. This enables later analysis by geographic area.

## What This Notebook Does

Input:
- 12 parquet files: yellow_tripdata_2024_01.parquet through yellow_tripdata_2024_12.parquet (Taxi Trip data for the whole of 2024)
- 1 CSV file: taxi_zone_lookup.csv (zone metadata)

Process:
1. Connect to PostgreSQL using SQLAlchemy
2. Read each parquet file into a pandas DataFrame
3. Insert data into PostgreSQL in chunks (100,000 rows at a time) for memory efficiency
4. Load the zone lookup table
5. Verify row counts and date ranges to see if everything is okay

Output:
- trips_raw table: approximately 36,300,000 rows (all 12 months combined)
- taxi_zones table: 265 rows (location reference data)

## Key Considerations

- The data is raw and has not been cleaned yet, strictly just loading in this notebook
- Large files are inserted in 100,000 row batches so there aren't any memory issues
- Tables are indexed on frequently queried columns like pickup time and Location ID's to improve query performance
- The zone lookup table is seperate and normalized to a different table to efficiently store the zone names

## 0. Imports & Configuration

In [ ]:
#   IMPORTS
import pandas as pd
import os
from sqlalchemy import create_engine, text
from pathlib import Path
import time



#   CONFIG
DB_PASSWORD = ''  #Put your database password
DB_USER     = ''  #Put your database username
DB_HOST     = 'localhost'   #Put your database host
DB_PORT     = '5432'  #Put your database port
DB_NAME     = ''   #Put the name of the database



#   PATHS
DATA_RAW_DIR       = Path('')   #Put the file path of the raw data
ZONE_LOOKUP_PATH   = Path('')   #Put the file path of the zone lookup table data

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

#   CoNNECTIONS TEST
with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print('ConnecteL:', result.scalar())

## 1. Load Zone Lookup → `taxi_zones`

In [ ]:
zones_df = pd.read_csv(ZONE_LOOKUP_PATH)

print(f'Shape:   {zones_df.shape}')
print(f'Columns: {zones_df.columns.tolist()}')
print()
zones_df.head()

In [ ]:
# LOAD ZONES TO POSTGRES
zones_df.to_sql('taxi_zones', con=engine, if_exists='replace', index=False)

with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM taxi_zones')).scalar()

print(f'loaded: {count} rows')

## 2. Load 12 Months of Trip Data → `trips_raw`

In [ ]:
# DATA FILES PREVIEW
parquet_files = sorted(DATA_RAW_DIR.glob('yellow_tripdata_2024-*.parquet'))
print(f'Found {len(parquet_files)} parquet files:')
for f in parquet_files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f'  {f.name}  ({size_mb:.0f} MB)')

if len(parquet_files) != 12:
    print('error')

In [ ]:
# INGESTION INTO POSTGRES
CHUNK_SIZE  = 100_000   
total_rows  = 0
start_time  = time.time()

for i, parquet_file in enumerate(parquet_files, start=1):
    t0 = time.time()
    df = pd.read_parquet(parquet_file)
    file_rows = len(df)
    mode = 'replace' if i == 1 else 'append'

    df.to_sql(
        'trips_raw',
        con=engine,
        if_exists=mode,
        index=False,
        chunksize=CHUNK_SIZE,
        method='multi'
    )

    total_rows += file_rows
    elapsed = time.time() - t0
    print(f'[{i:02d}/12] {parquet_file.name}  |  {file_rows:>8,} rows  |  {elapsed:.0f}s  |  running total: {total_rows:,}')

total_elapsed = time.time() - start_time
print(f'\n Done {total_rows:,} ')

## 3. Verify in PostgreSQL

In [ ]:
with engine.connect() as conn:
    trips_count = conn.execute(text('SELECT COUNT(*) FROM trips_raw')).scalar()
    zones_count = conn.execute(text('SELECT COUNT(*) FROM taxi_zones')).scalar()

print(f'trips_raw  → {trips_count:,} rows  (expected ~36M)')
print(f'taxi_zones → {zones_count:,} rows  (expected 265)')

# VERIFY IF ROW AMOUNT IS NORMAL
if trips_count < 30_000_000:
    print('low trip_raw rows')
else:
    print('Success')

In [ ]:
# CHECKING DATE RANGE
date_check = pd.read_sql_query('''
    SELECT
        MIN(tpep_pickup_datetime) AS earliest_pickup,
        MAX(tpep_pickup_datetime) AS latest_pickup,
        COUNT(DISTINCT DATE_TRUNC('month', tpep_pickup_datetime)) AS months_present
    FROM trips_raw
''', con=engine)

print('Date range:')
date_check

In [ ]:
# DATA PREVIEW
sample = pd.read_sql_query('SELECT * FROM trips_raw LIMIT 5', con=engine)
print('Sample Data:')
sample